# IR ALÉM 1 - Extração de Informações Clínicas com LLM
Este notebook demonstra o uso de Modelos de Linguagem de Grande Escala (LLM - simulado via API Claude ou Gemini localmente) para estruturação de anotações clínicas livres, alimentando o sistema de Assistente Conversacional (Fase 5) e integrando com o Classificador de ECG (Fase 4).

In [ ]:
import json
import os
import textwrap
from pydantic import BaseModel, Field
from typing import List, Optional

# Simulando a chamada LLM. Em produção, isso usaria anthropic.Anthropic() ou google.generativeai
def simular_chamada_llm(prompt: str, caso: str) -> str:
    """Moca o comportamento do LLM para execução local sem custo de API."""
    respostas = {
        "caso_1": '{"sintomas": ["palpitação", "coração acelerado"], "duracao": "1 hora", "intensidade": "moderada", "fatores_risco": ["histórico de arritmia"], "urgencia": "media", "classe_ecg_provavel": "S"}',
        "caso_2": '{"sintomas": ["dor no peito intensa", "falta de ar"], "duracao": "30 minutos", "intensidade": "alta", "fatores_risco": ["hipertensão", "tabagismo"], "urgencia": "alta", "classe_ecg_provavel": "V"}',
        "caso_3": '{"sintomas": ["exame de rotina"], "duracao": "N/A", "intensidade": "N/A", "fatores_risco": [], "urgencia": "baixa", "classe_ecg_provavel": "N"}'
    }
    
    if "dor no peito" in caso.lower():
        return respostas["caso_2"]
    elif "acelerado" in caso.lower() or "palpitação" in caso.lower():
        return respostas["caso_1"]
    else:
        return respostas["caso_3"]


## Definição do Schema de Saída (JSON)
O uso de frameworks como Pydantic garante que o output do LLM respeite o formato necessário para integração com o Watson Assistant e as classes da Fase 4.

In [ ]:
class ExtracaoClinica(BaseModel):
    sintomas: List[str] = Field(description="Lista de sintomas mencionados pelo paciente")
    duracao: str = Field(description="Duração dos sintomas (ex: 2 horas, 3 dias)")
    intensidade: str = Field(description="baixa, moderada, alta")
    fatores_risco: List[str] = Field(description="Condições pré-existentes como diabetes, pressão alta")
    urgencia: str = Field(description="baixa, media, alta")
    classe_ecg_provavel: Optional[str] = Field(description="Com base nos sintomas, se há possível correlação com classes F, N, Q, S ou V da Fase 4. Usar N se rotina.")

In [ ]:
def extrair_informacoes(texto_clinico: str) -> dict:
    prompt_system = f"""
    Você é um assistente médico especialista em cardiologia.
    Analise o texto do paciente e extraia as informações estruturadas estritamente no formato JSON abaixo.
    Mapeamento Fase 4 (CardioAI):
    - N (Normal): checkup, sem sintomas graves.
    - V (Ventricular): graves, risco iminente, dor no peito forte.
    - S (Supraventricular): palpitações, taquicardia leve a moderada.
    
    Schema JSON esperado:
    {ExtracaoClinica.schema_json()}
    """
    
    # Simulação da chamada (substituir por api_key de verdade na nuvem)
    resposta_json_str = simular_chamada_llm(prompt_system, texto_clinico)
    
    try:
        dados_estruturados = json.loads(resposta_json_str)
        return dados_estruturados
    except json.JSONDecodeError:
        return {"erro": "Falha no parse do JSON gerado pelo modelo"}


## Testes Práticos de Extração

In [ ]:
casos_uso = [
    "Doutor, meu coração está muito acelerado faz mais de uma hora. Já tive episódio de arritmia antes e estou preocupado.",
    "Estou com uma dor no peito muito forte, insuportável, irradiando para o braço esquerdo. Também sinto muita falta de ar. Sou fumante e hipertenso.",
    "Gostaria de agendar meu retorno anual. Fiz o ECG de rotina semana passada e queria mostrar os resultados."
]

for i, caso in enumerate(casos_uso, 1):
    print(f"\n{'='*50}")
    print(f"CASO {i}:")
    print(f"Texto Original: '{caso}'")
    resultado = extrair_informacoes(caso)
    print("\nDados Extraídos (JSON):")
    print(json.dumps(resultado, indent=2, ensure_ascii=False))
    
    # Integração Watson / Fase 4 (Demonstração lógica)
    print("\n--- Integração com Sistema CardioAI ---")
    if resultado.get("urgencia") == "alta":
        print("🚨 AÇÃO WATSON: Ativar dialog_node #emergencia -> Recomendar SAMU 192 imediato.")
    elif resultado.get("classe_ecg_provavel") == "S":
        print("💡 AÇÃO FASE 4: O sintoma sugere arritmia Supraventricular (S). O médico deve avaliar o ECG com prioridade moderada.")
    else:
        print("✅ AÇÃO WATSON: Ativar dialog_node #agendamento -> Seguir fluxo normal.")
